## Motion Exclusion HCP Data

*This script was used to identify subjects with excessive head motion under the lenient exclusion criteria (mean framewise displacement (mFD) > 0.55mm)  and the stringent exclusion criteria (mFD > .25 mm; FD > .2 mm less than 20%; no spikes > 5mm) suggested by Parkes et al. (2018).* 

### 1. Check how txt. files are set up

In [15]:
path = r'D:\Data Project 4\Motion Data\Movement_RelativeRMS_files\100610\MNINonLinear\Results\tfMRI_MOVIE1_7T_AP\Movement_RelativeRMS.txt'
with open(path, 'r', encoding= 'utf-8') as file:
    content = file.read()

print(content)

0
0.105353
0.184026
0.0974831
0.236087
0.0504996
0.122102
0.10688
0.280558
0.293082
0.0172796
0.238441
0.0946914
0.243266
0.292871
0.014596
0.299759
0.289995
0.0158627
0.309512
0.301155
0.0332869
0.368784
0.320632
0.0835765
0.326453
0.0809932
0.235875
0.193035
0.170805
0.280897
0.0632581
0.0930874
0.181577
0.366968
0.135969
0.311818
0.370045
0.0527306
0.332493
0.0998506
0.252408
0.152313
0.241086
0.334236
0.0159364
0.28393
0.147746
0.164776
0.169923
0.175304
0.325612
0.121749
0.0742831
0.249367
0.400293
0.262983
0.0403127
0.302595
0.359953
0.115792
0.249581
0.297697
0.0720163
0.296234
0.227177
0.125476
0.327636
0.17139
0.153031
0.331927
0.11858
0.374121
0.356197
0.0872606
0.387406
0.233436
0.187147
0.342318
0.146466
0.227783
0.21127
0.104077
0.329936
0.0983569
0.0515876
0.0783852
0.177008
0.0477283
0.31005
0.401274
0.0380473
0.357933
0.217565
0.207832
0.380117
0.0368364
0.329533
0.155604
0.167932
0.301175
0.0803248
0.249514
0.289507
0.0261116
0.284211
0.28389
0.0404594
0.290292
0.25335

### 2. Test: Compute mean of values that are saved in text file

In [3]:
# absolut path to data file
file_path = r'E:\Data Project 4\Motion Data\Movement_RelativeRMS_files\100610\MNINonLinear\Results\tfMRI_MOVIE1_7T_AP\Movement_RelativeRMS.txt'
 
# read in values as numbers
with open(file_path, 'r', encoding='utf-8') as file:
    numbers = [float(line.strip()) for line in file]  # Convert Lines in Floats 
 
# Compute average
meanFD = sum(numbers) / len(numbers)
 
# print average
print(f"meanFD is: {meanFD}")

meanFD is: 0.25677917491856656


### 3. Find subjects to exclude based on Parkes lenient criteria (mFD > 0.55)

In [7]:
import os
from pathlib import Path
import pandas as pd

# base path
base_path = Path(r'E:\Data Project 4\Motion Data\Movement_RelativeRMS_files')

# list of relevant movie folders
relevant_folders = {
    "tfMRI_MOVIE1_7T_AP",
    "tfMRI_MOVIE2_7T_PA",
    "tfMRI_MOVIE3_7T_PA",
    "tfMRI_MOVIE4_7T_AP",
}

# list for results
results = []

# Iteration across all subject folders 
for subject_folder in base_path.iterdir():
    if subject_folder.is_dir(): 
        # path to MNINonLinear/Results folder
        results_path = subject_folder / "MNINonLinear" / "Results"
        
        if results_path.exists():  # check if results folder exists
            # iteration across all subfolders in the results folder
            for movie_folder in results_path.iterdir():
                if movie_folder.is_dir() and movie_folder.name in relevant_folders:  # filter relevant movie folders
                    # file Movement_RelativeRMS.txt in sub-folder
                    file_path = movie_folder / "Movement_RelativeRMS.txt"
                    
                    if file_path.exists():  # Check if file exists
                        try:
                            # read file and convert to numbers 
                            with open(file_path, 'r', encoding='utf-8') as file:
                                numbers = [float(line.strip()) for line in file]
                            
                            mean_fd = sum(numbers) / len(numbers)  # compute average 
                            
                            # save results 
                            results.append({
                                "Subject": subject_folder.name,
                                "MovieFolder": movie_folder.name,
                                "MeanFD": mean_fd
                            })
                        except Exception as e:
                            print(f"Error when processing file: {file_path} – {e}")

# results as DataFrame
df = pd.DataFrame(results)

# save data 
output_path = Path(r'E:\Data Project 4\Motion Data\meanFD_results.csv')
df.to_csv(output_path, index=False)

# filter averages above 0.55 mm
high_mean_fd = df[df["MeanFD"] > 0.55]

# print result
print("mFD above 0.55:")
print(high_mean_fd)

# optional: save filtered data 
high_mean_fd.to_csv(output_path.with_name("meanFD_high.csv"), index=False)


mFD above 0.55:
Empty DataFrame
Columns: [Subject, MovieFolder, MeanFD]
Index: []


### 4. Find subjects to exclude based on Parkes stringent criteria (mFD > .25 mm; FD > .2 mm less than 20%; no spikes > 5mm)

In [18]:
import os
from pathlib import Path
import pandas as pd

# Base path
base_path = Path(r'E:\Data Project 4\Motion Data\Movement_RelativeRMS_files')

# List of relevant movie folders
relevant_folders = {
    "tfMRI_MOVIE1_7T_AP",
    "tfMRI_MOVIE2_7T_PA",
    "tfMRI_MOVIE3_7T_PA",
    "tfMRI_MOVIE4_7T_AP",
}

# list for results
results = []

# Count how many subjects have all movie folders
subjects_with_all_movies = 0

# Iteration across all subject folders 
for subject_folder in base_path.iterdir():
    if subject_folder.is_dir():  
        # Path to MNINonLinear/Results folder
        results_path = subject_folder / "MNINonLinear" / "Results"
        
        if results_path.exists():  # check if results folder exists
            found_movies = set()  # set to save found movie folders
            
            # Iteration across all subfiles in results folder
            for movie_folder in results_path.iterdir():
                if movie_folder.is_dir() and movie_folder.name in relevant_folders:  # filter for relevant movie files
                    # file Movement_RelativeRMS.txt in subfolder
                    file_path = movie_folder / "Movement_RelativeRMS.txt"
                    
                    if file_path.exists():  # check if file exists
                        try:
                            # read file
                            with open(file_path, 'r', encoding='utf-8') as file:
                                numbers = [float(line.strip()) for line in file]
                            
                            # compute average 
                            mean_fd = sum(numbers) / len(numbers)
                            
                            # single fd values > 5.00
                            has_value_over_5 = any(x > 5.00 for x in numbers)
                            
                            # proportion of values > 0.2
                            proportion_over_0_2 = sum(x > 0.2 for x in numbers) / len(numbers)
                            has_20_percent_over_0_2 = proportion_over_0_2 >= 0.2
                            
                            # save results if one criteria is fulfilled
                            if mean_fd > 0.25 or has_value_over_5 or has_20_percent_over_0_2:
                                results.append({
                                    "Subject": subject_folder.name,
                                    "MovieFolder": movie_folder.name,
                                    "MeanFD": mean_fd,
                                    "ValueOver5": has_value_over_5,
                                    "20PercentOver0.2": has_20_percent_over_0_2
                                })
                            
                            # add movie folder to 'found movies'
                            found_movies.add(movie_folder.name)
                            
                        except Exception as e:
                            print(f"error when processing file: {file_path} – {e}")
            
            # check whether all four movie files were found 
            if found_movies == relevant_folders:
                subjects_with_all_movies += 1

# results as DataFrame
df = pd.DataFrame(results)

# number of subjects with fulfilled criteria (just count as 1 if more than 1 criteria is positive)
unique_subjects = df['Subject'].nunique()

# print number of unique subjects and subjects with all 4 movie folders
print(f"number of unique subjects: {unique_subjects}")
print(f"number of subjects with all 4 movie folders: {subjects_with_all_movies}")

# Save results 
output_path = Path(r'E:\Data Project 4\Motion Data\filtered_results.csv')
df.to_csv(output_path, index=False)

# Print results
print("filtered results:")
print(df)

number of unique subjects: 62
number of subjects with all 4 movie folders: 178
filtered results:
    Subject         MovieFolder    MeanFD  ValueOver5  20PercentOver0.2
0    100610  tfMRI_MOVIE1_7T_AP  0.256779       False              True
1    100610  tfMRI_MOVIE2_7T_PA  0.178953       False              True
2    100610  tfMRI_MOVIE3_7T_PA  0.201716       False              True
3    100610  tfMRI_MOVIE4_7T_AP  0.240089       False              True
4    102816  tfMRI_MOVIE1_7T_AP  0.226994       False              True
..      ...                 ...       ...         ...               ...
139  927359  tfMRI_MOVIE1_7T_AP  0.166982       False              True
140  927359  tfMRI_MOVIE2_7T_PA  0.150667       False              True
141  927359  tfMRI_MOVIE3_7T_PA  0.180407       False              True
142  927359  tfMRI_MOVIE4_7T_AP  0.191026       False              True
143  995174  tfMRI_MOVIE3_7T_PA  0.173947       False              True

[144 rows x 5 columns]
